In [ ]:
from __future__ import annotations
from egglog import *


class Num(Expr):
    def __init__(self, value: i64Like) -> None: ...

    @classmethod
    def var(cls, name: StringLike) -> Num:  ...

    def __add__(self, other: Num) -> Num: ...

    def __mul__(self, other: Num) -> Num: ...

egraph = EGraph()

expr1 = egraph.let("expr1", Num(2) * (Num.var("x") + Num(3)))
expr2 = egraph.let("expr2", Num(6) + Num(2) * Num.var("x"))

@egraph.register
def _num_rule(a: Num, b: Num, c: Num, i: i64, j: i64):
    yield rewrite(a + b).to(b + a)
    yield rewrite(a * (b + c)).to((a * b) + (a * c))
    yield rewrite(Num(i) + Num(j)).to(Num(i + j))
    yield rewrite(Num(i) * Num(j)).to(Num(i * j))

egraph.saturate()
egraph.check(expr1 == expr2)
egraph.extract(expr1)


Num(2) * (Num.var("x") + Num(3))

In [2]:
from __future__ import annotations
from egglog import *

egraph = EGraph()

@egraph.class_
class Shape(Expr):
    def __init__(self, name: StringLike) -> None: ...
    
    @classmethod
    def Constant(cls, value: f64Like) -> Shape: ...
    
    # Example of a "Other" function from your Rust code
    def Add(self, other: Shape) -> Shape: ...
    def Mul(self, other: Shape) -> Shape: ...

# In egglog-python, cost is handled by the extractor automatically, 
# but you can influence it by defining rules that favor certain forms.
@egraph.register
def rules(a: Shape, b: Shape, i: f64, j: f64):
    # Constant folding (Equivalent to your Rust math rules)
    yield rewrite(Shape.Constant(i) + Shape.Constant(j)).to(Shape.Constant(i + j))
    # Commutativity
    yield rewrite(a + b).to(b + a)

# To mimic your 'egrid_val_map' analysis, we use 'functions' in egglog
@egraph.function
def shape_value(s: Shape) -> f64: ...

# Now you can use it
expr = egraph.let("e", Shape.Constant(2.0) + Shape.Constant(3.0))
egraph.run(10)
print(egraph.extract(expr))

AttributeError: 'EGraph' object has no attribute 'class_'

In [5]:
from __future__ import annotations
from egglog import *

# Initialize your EGraph
egraph = EGraph()

# Define your DSL
class Shape(Expr):
    def __init__(self, name: StringLike) -> None: ...
    
    @classmethod
    def Cuboid(cls, w: f64Like, h: f64Like, d: f64Like) -> Shape: ...
    
    def Union(self, other: Shape) -> Shape: ...
    def Move(self, x: f64Like, y: f64Like, z: f64Like) -> Shape: ...
    def SymRef(self, axis: StringLike) -> Shape: ...
    def Inv(self) -> Shape: ...

def Add(a: f64Like, b: f64Like) -> f64: ...

In [15]:
def shapecoder_rules(
    s: Shape, s2: Shape,
    x1: f64, y1: f64, z1: f64,
    x2: f64, y2: f64, z2: f64,
    w: f64, h: f64, d: f64
):
    # 1. Commutativity: Union(A, B) == Union(B, A)
    # This ensures the order of the cuboids doesn't break the match
    yield rewrite(s.Union(s2)).to(s2.Union(s))

    # 2. X-axis symmetry
    # Note: Use the same variables (w, h, d) for both cuboids 
    # so egglog knows they must be identical.
    yield rewrite(
        Shape.Cuboid(w, h, d).Move(x1, y1, z1).Union(
            Shape.Cuboid(w, h, d).Move(x2, y1, z1)
        )
    ).to(
        Shape.Cuboid(w, h, d).Move(x1, y1, z1).SymRef("AX")
    ).when(
        (x1 + x2).abs() < 0.01 
    )

In [18]:
egraph = EGraph()

# Define the input again
c1 = Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0)
c2 = Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0)
test_shape = c1.Union(c2)

# Run
res = egraph.let("design", test_shape)
egraph.run(20) # Increased iterations to let commutativity "settle"

# Extract
optimized = egraph.extract(res)
print("Optimized Result:", optimized)

Optimized Result: Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0).Union(Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0))


In [20]:
from __future__ import annotations
from egglog import *

egraph = EGraph()


class Shape(Expr):
    def __init__(self, name: StringLike) -> None: ...
    @classmethod
    def Cuboid(cls, w: f64Like, h: f64Like, d: f64Like) -> Shape: ...
    def Union(self, other: Shape) -> Shape: ...
    def Move(self, x: f64Like, y: f64Like, z: f64Like) -> Shape: ...
    def SymRef(self, axis: StringLike) -> Shape: ...

# --- THE RULES ---

def shapecoder_logic(
    s1: Shape, s2: Shape,
    w: f64, h: f64, d: f64,
    x1: f64, y1: f64, z1: f64,
    x2: f64, y2: f64, z2: f64
):
    # 1. Define what a mirror is (The condition)
    is_mirror_x = (x1 + x2).abs() < 0.01
    same_y_z = (y1 == y2) & (z1 == z2)
    
    # 2. The Rewrite Rule
    # We use a explicit pattern for the Union of two moved Cuboids
    yield rewrite(
        Shape.Cuboid(w, h, d).Move(x1, y1, z1).Union(
            Shape.Cuboid(w, h, d).Move(x2, y2, z2)
        )
    ).to(
        Shape.Cuboid(w, h, d).Move(x1, y1, z1).SymRef("AX")
    ).when(is_mirror_x, same_y_z)

    # 3. Add Commutativity for Union (Crucial!)
    yield rewrite(s1.Union(s2)).to(s2.Union(s1))

# --- THE TEST ---
# Create input with explicit floats
c1 = Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0)
c2 = Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0)
test_design = c1.Union(c2)

# Run equality saturation
design_expr = egraph.let("design", test_design)
egraph.run(20)

# Extract the best version
optimized = egraph.extract(design_expr)
print("Optimized Result:", optimized)

Optimized Result: Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0).Union(Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0))


In [22]:
from __future__ import annotations
from egglog import *

egraph = EGraph()


class Shape(Expr):
    def __init__(self, name: StringLike) -> None: ...
    @classmethod
    def Cuboid(cls, w: f64Like, h: f64Like, d: f64Like) -> Shape: ...
    def Union(self, other: Shape) -> Shape: ...
    def Move(self, x: f64Like, y: f64Like, z: f64Like) -> Shape: ...
    def SymRef(self, axis: StringLike) -> Shape: ...


def shapecoder_logic(
    w: f64, h: f64, d: f64,
    x1: f64, y1: f64, z1: f64,
    x2: f64, y2: f64, z2: f64
):
    # Match the pattern of two cuboids inside a union
    # We use explicit variables for everything to avoid matching errors
    left_cuboid = Shape.Cuboid(w, h, d).Move(x1, y1, z1)
    right_cuboid = Shape.Cuboid(w, h, d).Move(x2, y2, z2)
    combined = left_cuboid.Union(right_cuboid)
    
    # Define the symmetry version
    symmetry_version = left_cuboid.SymRef("AX")

    # Use 'yield union().if_()' to force them into the same equivalence class
    yield union(combined).with_(symmetry_version).if_(
        (x1 + x2).abs() < 0.01,
        y1 == y2,
        z1 == z2
    )

# --- EXECUTION ---
c1 = Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0)
c2 = Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0)
test_design = c1.Union(c2)

design_expr = egraph.let("design", test_design)
egraph.run(20)

# Check if the engine actually "sees" the symmetry
print("Does the e-graph know they are equal?")
print(egraph.check(test_design == c1.SymRef("AX"))) 

# Extract
optimized = egraph.extract(design_expr)
print("\nOptimized Result:", optimized)

Does the e-graph know they are equal?


EggSmolError: At 0:0 of 
Check failed: 
(= (Shape_Union (Shape_Move (Shape_Cuboid 2.0 2.0 2.0) 1.0 0.0 0.0) (Shape_Move (Shape_Cuboid 2.0 2.0 2.0) -1.0 0.0 0.0)) (Shape_SymRef (Shape_Move (Shape_Cuboid 2.0 2.0 2.0) 1.0 0.0 0.0) "AX"))

In [23]:
# Before running egraph.run()
egraph.register(Shape.Cuboid(2.0, 2.0, 2.0))

In [24]:
from __future__ import annotations
from egglog import *

# 1. Initialize EGraph
egraph = EGraph()

# 2. Define the DSL (Shapes and Math)

class Shape(Expr):
    def __init__(self, name: StringLike) -> None: ...
    
    @classmethod
    def Cuboid(cls, w: f64Like, h: f64Like, d: f64Like) -> Shape: ...
    
    def Union(self, other: Shape) -> Shape: ...
    def Move(self, x: f64Like, y: f64Like, z: f64Like) -> Shape: ...
    def SymRef(self, axis: StringLike) -> Shape: ...

# 3. Define the Shapecoder Rules

def shapecoder_rules(
    s1: Shape, s2: Shape,
    x1: f64, y1: f64, z1: f64,
    x2: f64, y2: f64, z2: f64
):
    # Rule A: Commutativity of Union 
    # (Allows the engine to see Union(A, B) and Union(B, A) as identical)
    yield rewrite(s1.Union(s2)).to(s2.Union(s1))

    # Rule B: Symmetry Detection (X-Axis)
    # We match any two moved shapes and check if they are identical and mirrored
    yield rule(
        s1.Move(x1, y1, z1).Union(s2.Move(x2, y2, z2))
    ).then(
        union(s1.Move(x1, y1, z1).Union(s2.Move(x2, y2, z2))).with_(
            s1.Move(x1, y1, z1).SymRef("AX")
        )
    ).if_(
        s1 == s2,            # The base shapes are the same
        (x1 + x2).abs() < 0.01, # X coordinates are mirrored
        y1 == y2,            # Y is the same
        z1 == z2             # Z is the same
    )

# 4. Define the Test Case
# IMPORTANT: Use .0 to ensure these are f64, not integers
cuboid_a = Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0)
cuboid_b = Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0)
complex_shape = cuboid_a.Union(cuboid_b)

# 5. Run the E-Graph
# We 'let' a variable hold the expression so we can extract it later
design = egraph.let("design", complex_shape)

print("Running equality saturation...")
egraph.run(20)

# 6. Extract and Verify
try:
    # Check if the engine actually proved the equality
    # This will throw an error if the rule didn't fire
    egraph.check(complex_shape == Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0).SymRef("AX"))
    print("Proof Successful: Symmetry recognized!")
except Exception as e:
    print("Proof Failed. Check egraph.display() to see the nodes.")

optimized = egraph.extract(design)
print("\n--- Optimized Result ---")
print(optimized)

# 7. Visualize (Optional)
# egraph.display()

Running equality saturation...
Proof Failed. Check egraph.display() to see the nodes.

--- Optimized Result ---
Shape.Cuboid(2.0, 2.0, 2.0).Move(1.0, 0.0, 0.0).Union(Shape.Cuboid(2.0, 2.0, 2.0).Move(-1.0, 0.0, 0.0))
